# 04 · Grad-CAM++: where does each model look, and can we trust it?

Phase 2. Every trained model is analysed on the **same** test images from FF++ and the unseen Celeb-DF:

1. **Run the analysis:** Grad-CAM, Grad-CAM++ and a random map for every model and image. It measures **faithfulness**
   (deletion / insertion, Petsiuk et al. 2018) and **which facial regions** hold the evidence. Regions are placed per face
   from the detector's 5 landmarks.
2. **Overview:** region placement check, heatmap gallery, faithfulness across models, region attribution across models.
3. **Grad-CAM profile of each model**, the same report for every model, as in notebook 03.

**Notebook settings (right sidebar)**
- **Accelerator:** GPU T4 x2 · **Internet:** On
- **Inputs:** **01_extract_faces** (FF++), **01b_extract_celebdf**, **02_train** (plus any copies of 02).
  Optional: **03_evaluate**, to list models in its ranking order.

Run with **Run All**, or commit with **Save & Run All (Commit)**. Everything runs top to bottom in one session.

In [ ]:
# --- Project code -------------------------------------------------------------------------
# Always clones the latest code. It lives in /tmp, OUTSIDE /kaggle/working, so it is never saved into this
# notebook's output (otherwise later notebooks would pick up a stale copy). For a private repo add a
# GITHUB_TOKEN secret. Fallback: a zip of the project uploaded as a Kaggle *dataset*.
import os, sys, glob, shutil, subprocess

PROJECT = "/tmp/dfdc-project"
REPO, BRANCH = "https://github.com/Amay-M-Nair/DFDC.git", "dev"

if not os.path.exists(PROJECT):
    token = ""
    try:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret("GITHUB_TOKEN")
    except Exception:
        pass
    url = REPO.replace("https://", f"https://{token}@") if token else REPO
    r = subprocess.run(["git", "clone", "-q", "-b", BRANCH, url, PROJECT], capture_output=True)
    if r.returncode == 0:
        subprocess.run(["git", "-C", PROJECT, "remote", "set-url", "origin", REPO])  # don't keep the token
    else:
        found = [p for d in range(5) for p in glob.glob("/kaggle/input/" + "*/" * d + "src/config.py")]
        if not found:
            raise RuntimeError("git clone failed: add a GITHUB_TOKEN secret or attach the project zip as a dataset")
        print("WARNING: git clone failed; using the attached copy at", os.path.dirname(os.path.dirname(found[0])))
        shutil.copytree(os.path.dirname(os.path.dirname(found[0])), PROJECT,
                        ignore=shutil.ignore_patterns("data", "outputs", "__pycache__"))

shutil.rmtree("/kaggle/working/project", ignore_errors=True)  # remove stale copies saved by older notebook versions
os.chdir(PROJECT)
sys.path.insert(0, PROJECT)
!pip install -q grad-cam anthropic
!git -C {PROJECT} log -1 --format="code version: %h  %s  (%cd)" --date=short 2>/dev/null || echo "code version: attached copy (no git)"

In [ ]:
%matplotlib inline
import glob, inspect, os
from pathlib import Path
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from IPython.display import display, Markdown, Image as Show
from pytorch_grad_cam.utils.image import show_cam_on_image

from src.kaggle_utils import setup, find_datasets, checkpoint_roots, find_all
setup(data=False)

import src.xai.evaluate_cam as evaluate_cam
assert "heatmaps_" in inspect.getsource(evaluate_cam), (
    "The cloned code is outdated (no per-model heatmap saving). Restart the session (Run → Restart) and run all again.")

from src.classifier.compare import find_checkpoints
from src.classifier.model import BASELINES
from src.config import IMG_SIZE
from src.faces import FaceCropper
from src.xai.gradcam import Explainer, FACE_REGIONS, BACKGROUND, draw_bbox, draw_regions, landmark_regions

DATASETS = {k: v for k, v in find_datasets().items() if k in ("FF++", "CelebDF")}
ROOTS = checkpoint_roots()
CKPTS = find_checkpoints(ROOTS)
print("datasets:", DATASETS)
print("models:", list(CKPTS))
assert "FF++" in DATASETS, "Add the output of 01_extract_faces as an input"
assert CKPTS, "Add the output of 02_train as an input"
if "CelebDF" not in DATASETS:
    print("WARNING: Celeb-DF crops not attached; only FF++ will be analysed.")

OUT = "/kaggle/working/outputs/gradcam"
os.makedirs(OUT, exist_ok=True)
REGION_COLS = list(FACE_REGIONS) + [BACKGROUND]
SHORT = {"left eye": "L eye", "right eye": "R eye", "nose": "nose", "mouth / teeth": "mouth",
         "left cheek": "L cheek", "right cheek": "R cheek", "forehead / hairline": "forehead",
         "chin / jawline": "chin", BACKGROUND: "border / bg"}

## Settings
`N` = images per dataset (one frame per test video, balanced across manipulation methods). Every model sees the same images. About 3–6 minutes per model at `N = 300`.

In [ ]:
N = 300
CAM_METHODS = ["gradcam++", "gradcam", "random"]

## 1 · Run the analysis
Computes heatmaps, faithfulness curves and region attribution for every model, and saves each model's heatmaps for the profiles in section 3.

In [ ]:
roots = " ".join(f'"{r}"' for r in ROOTS)
datasets = " ".join(f'"{k}={v}"' for k, v in DATASETS.items())
!TQDM_MININTERVAL=60 python -m src.xai.evaluate_cam --roots {roots} --datasets {datasets} --n {N} --methods {" ".join(CAM_METHODS)}

needed = ["faithfulness.csv", "faithfulness_summary.csv", "faithfulness_tests.csv", "curves.csv", "regions.csv", "region_attribution.csv"]
missing = [f for f in needed if not os.path.exists(f"{OUT}/{f}")]
HEATMAPS = {str(np.load(f)["backbone"]): f for f in sorted(glob.glob(f"{OUT}/heatmaps_*.npz"))}
assert not missing, f"The analysis did not finish (missing {missing}). Scroll up for the error."
assert HEATMAPS, "No per-model heatmap files were saved. Scroll up for the error."
print("saved heatmaps for:", list(HEATMAPS))

faith = pd.read_csv(f"{OUT}/faithfulness.csv")
faith = faith[~faith.empty_cam.astype(bool)]
summary = pd.read_csv(f"{OUT}/faithfulness_summary.csv")
tests = pd.read_csv(f"{OUT}/faithfulness_tests.csv")
tests["significant"] = tests.p_value < 0.05
curves_df = pd.read_csv(f"{OUT}/curves.csv")
regions_df = pd.read_csv(f"{OUT}/regions.csv")
attribution = pd.read_csv(f"{OUT}/region_attribution.csv")

hits = find_all(os.path.join("compare", "summary.csv"))      # notebook 03's ranking, if attached
ORDER = [b for b in (pd.read_csv(hits[0]).backbone if hits else CKPTS) if b in HEATMAPS]
ORDER += [b for b in HEATMAPS if b not in ORDER]
print("profile order:", ORDER)

## 2 · Overview

### Facial-region placement (sanity check)
Region boxes from the 5 landmarks on random test faces. The eye, nose and mouth boxes should sit on those features.
Colours: left eye red · right eye orange · nose yellow · mouth green · cheeks blue · forehead purple · chin white.

In [ ]:
faces = FaceCropper()
picks = []
for ds, root in DATASETS.items():
    files = sorted(Path(root, "test").glob("*/*.jpg"))
    picks += [(ds, files[i]) for i in np.random.default_rng(0).choice(len(files), 5, replace=False)]
fig, axes = plt.subplots(1, len(picks), figsize=(2.4 * len(picks), 2.8))
for ax, (ds, p) in zip(np.atleast_1d(axes), picks):
    rgb = np.asarray(Image.open(p).convert("RGB").resize((IMG_SIZE, IMG_SIZE)))
    found = faces.landmarks(rgb)
    regions = landmark_regions(*found, IMG_SIZE, IMG_SIZE) if found is not None else FACE_REGIONS
    ax.imshow(draw_regions(rgb, regions)); ax.axis("off")
    ax.set_title(f"{ds}\n{'landmarks' if found is not None else 'grid (no face)'}", fontsize=8)
plt.tight_layout(); plt.savefig(f"{OUT}/region_placement.png", dpi=130); plt.show()
print(f"regions from landmarks on {(regions_df.region_source == 'landmarks').mean():.0%} of analysed maps")

### Heatmap gallery
Same images for every model: one fake per FF++ manipulation, one FF++ real, and a Celeb-DF real and fake. Title = verdict, P(fake), region with most activation.

In [ ]:
gallery = []
for ds, root in DATASETS.items():
    by_method = {}
    for p in sorted(Path(root, "test").glob("*/*.jpg")):
        by_method.setdefault(p.name.split("__")[0], []).append(p)
    rng = np.random.default_rng(1)
    gallery += [(ds, m, ps[rng.integers(len(ps))]) for m, ps in sorted(by_method.items())]

fig, axes = plt.subplots(len(ORDER) + 1, len(gallery), figsize=(2.3 * len(gallery), 2.5 * (len(ORDER) + 1)))
for j, (ds, m, p) in enumerate(gallery):
    axes[0, j].imshow(Image.open(p).resize((IMG_SIZE, IMG_SIZE)))
    axes[0, j].set_title(f"{ds}\n{'real' if m == 'original' else m}", fontsize=8)
for i, backbone in enumerate(ORDER, start=1):
    explain = Explainer(CKPTS[backbone])
    for j, (ds, m, p) in enumerate(gallery):
        e = explain(Image.open(p))
        axes[i, j].imshow(draw_bbox(e.overlay, e.bbox))
        axes[i, j].set_title(f"{e.label} {e.prob_fake:.2f} · {SHORT.get(e.top_regions[0][0])}", fontsize=7)
    axes[i, 0].set_ylabel(BASELINES.get(backbone, backbone), fontsize=8)
    del explain
for ax in axes.flat:
    ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout(); plt.savefig(f"{OUT}/gallery.png", dpi=130); plt.show()

### Faithfulness across models
**Deletion AUC: lower is better** (removing the highlighted pixels quickly kills the model's confidence). **Insertion AUC: higher is better.** A method is only meaningful if it beats `random`.

In [ ]:
for metric in ("deletion_auc", "insertion_auc"):
    display(Markdown(f"**{metric}**"))
    display(summary.pivot_table(index=["dataset", "model"], columns="cam", values=metric).round(4))
display(Markdown("**Paired Wilcoxon tests** (same images; p < 0.05 = significant)"))
display(tests.round(4))

### Region attribution across models
Mean share of Grad-CAM++ activation per facial region (correct predictions). Rows = models; one panel per manipulation.

In [ ]:
regions = [c for c in attribution.columns if c not in ("model", "dataset", "manipulation")]
panels = attribution[["dataset", "manipulation"]].drop_duplicates().values.tolist()
fig, axes = plt.subplots(len(panels), 1, figsize=(11, 2.6 * len(panels)))
for ax, (ds, manip) in zip(np.atleast_1d(axes), panels):
    t = attribution[(attribution.dataset == ds) & (attribution.manipulation == manip)].set_index("model")[regions]
    ax.imshow(t.values, cmap="magma", vmin=0, vmax=max(0.4, t.values.max()))
    ax.set_yticks(range(len(t)), t.index, fontsize=8)
    ax.set_xticks(range(len(regions)), [SHORT.get(r, r) for r in regions], fontsize=8)
    for (r, c), v in np.ndenumerate(t.values):
        ax.text(c, r, f"{v:.2f}", ha="center", va="center", fontsize=7, color="white" if v < 0.25 else "black")
    ax.set_title(f"{ds} · {'real' if manip == 'original' else manip}", fontsize=9)
plt.tight_layout(); plt.savefig(f"{OUT}/region_attribution.png", dpi=130); plt.show()

## 3 · Grad-CAM profile of each model
The same report for every model, built from the heatmaps saved in section 1 (nothing is recomputed):
- **Faithfulness:** deletion / insertion per dataset for Grad-CAM, Grad-CAM++ and random, the model's significance tests, and its curves
- **Region attribution:** where the evidence is, for each manipulation (correct predictions)
- **Average heatmap:** mean Grad-CAM++ map per manipulation over the average face
- **Grad-CAM vs. Grad-CAM++:** the same correctly classified fakes under both methods
- **Mistakes:** most confident mistakes on Celeb-DF with the Grad-CAM++ map and its top region

In [ ]:
def up(h):
    return cv2.resize(h.astype(np.float32), (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_CUBIC).clip(0, 1)

def face(path):
    return np.asarray(Image.open(path).convert("RGB").resize((IMG_SIZE, IMG_SIZE)), dtype=np.float32) / 255.0

def overlay(rgb, h):
    return show_cam_on_image(rgb.astype(np.float32), up(h), use_rgb=True)

def group_name(ds, manip):
    return f"{ds} {'real' if manip == 'original' else manip}"

def profile(backbone):
    z = np.load(HEATMAPS[backbone])
    name = str(z["model"])
    meta = pd.DataFrame({k: z[k].astype(str) for k in ("dataset", "files", "path", "manipulation", "label", "pred")})
    meta["prob"] = z["prob"]
    meta["correct"] = meta.label == meta.pred
    cams = {k[4:].replace("pp", "++"): z[k] for k in z.files if k.startswith("cam_")}
    reg = regions_df[regions_df.model == name]
    acc = meta.groupby("dataset").correct.mean()
    display(Markdown(f"---\n### {name}\nGrad-CAM layer `{z['target_layer']}` · threshold {float(z['threshold']):.2f} · "
                     + " · ".join(f"{d} accuracy on analysed images {a:.0%}" for d, a in acc.items())))

    # Faithfulness
    f = faith[faith.model == name].groupby(["dataset", "cam"])[["deletion_auc", "insertion_auc"]].mean().unstack("cam")
    display(Markdown("**Faithfulness** (deletion ↓ better · insertion ↑ better)")); display(f.round(4))
    display(tests[tests.model == name].drop(columns="model").round(4))
    c = curves_df[curves_df.model == name]
    dss = list(c.dataset.unique())
    fig, axes = plt.subplots(1, 2 * len(dss), figsize=(4 * 2 * len(dss), 3.1))
    axes = np.atleast_1d(axes)
    for j, ds in enumerate(dss):
        for cam, g in c[c.dataset == ds].groupby("cam"):
            axes[2 * j].plot(g.fraction, g.deletion, label=cam)
            axes[2 * j + 1].plot(g.fraction, g.insertion, label=cam)
        axes[2 * j].set_title(f"{ds} deletion (↓)", fontsize=9); axes[2 * j + 1].set_title(f"{ds} insertion (↑)", fontsize=9)
    for ax in axes: ax.set_xlabel("fraction of pixels"); ax.set_ylim(0, 1.02)
    axes[0].set_ylabel("P(predicted class)"); axes[0].legend(fontsize=8)
    plt.tight_layout(); plt.show()

    # Region attribution
    r = reg[(reg.cam == "gradcam++") & reg.correct.astype(bool)]
    if len(r):
        att = r.groupby(["dataset", "manipulation"])[REGION_COLS].mean()
        att.index = [group_name(*i) for i in att.index]
        ax = att.rename(columns=SHORT).T.plot.bar(figsize=(13, 3.4), rot=0, width=0.85)
        ax.set_ylabel("share of Grad-CAM++"); ax.set_title(f"{name}: where the evidence is", fontsize=10)
        ax.legend(fontsize=7, ncol=4); plt.tight_layout(); plt.show()

    # Average heatmap per manipulation (correct predictions)
    groups = list(meta[meta.correct].groupby(["dataset", "manipulation"]))
    if groups:
        fig, axes = plt.subplots(1, len(groups), figsize=(2.2 * len(groups), 2.6))
        for ax, ((ds, manip), g) in zip(np.atleast_1d(axes), groups):
            idx = g.index[:40]
            mean_face = np.mean([face(p) for p in meta.path[idx]], axis=0)
            mean_map = cams["gradcam++"][idx].astype(np.float32).mean(0)
            ax.imshow(overlay(mean_face, mean_map / max(float(mean_map.max()), 1e-8))); ax.axis("off")
            ax.set_title(f"{group_name(ds, manip)}\n(n={len(g)})", fontsize=8)
        plt.suptitle(f"{name}: average Grad-CAM++ map (correct predictions)", fontsize=10); plt.tight_layout(); plt.show()

    # Grad-CAM vs Grad-CAM++ on the same correctly classified fakes
    ex = meta[meta.correct & (meta.label == "fake") & (meta.dataset == "FF++")].groupby("manipulation").head(1).index[:5]
    if len(ex) and "gradcam" in cams:
        fig, axes = plt.subplots(3, len(ex), figsize=(2.3 * len(ex), 7))
        axes = np.asarray(axes).reshape(3, len(ex))
        for j, i in enumerate(ex):
            rgb = face(meta.path[i])
            axes[0, j].imshow(rgb); axes[0, j].set_title(f"{meta.manipulation[i]}\nP(fake)={meta.prob[i]:.2f}", fontsize=8)
            axes[1, j].imshow(overlay(rgb, cams["gradcam"][i]))
            axes[2, j].imshow(overlay(rgb, cams["gradcam++"][i]))
        axes[1, 0].set_ylabel("Grad-CAM"); axes[2, 0].set_ylabel("Grad-CAM++")
        for ax in axes.flat: ax.set_xticks([]); ax.set_yticks([])
        plt.suptitle(f"{name}: Grad-CAM vs. Grad-CAM++", fontsize=10); plt.tight_layout(); plt.show()

    # Most confident mistakes (Celeb-DF if attached)
    ds = "CelebDF" if "CelebDF" in set(meta.dataset) else "FF++"
    m = meta[(meta.dataset == ds) & ~meta.correct].copy()
    m["confidence"] = np.where(m.pred == "fake", m.prob, 1 - m.prob)
    worst = m.nlargest(8, "confidence")
    if len(worst):
        top = reg[reg.cam == "gradcam++"].drop_duplicates(["dataset", "file"]).set_index(["dataset", "file"])[REGION_COLS]
        fig, axes = plt.subplots(1, len(worst), figsize=(2.3 * len(worst), 2.9))
        for ax, (i, w) in zip(np.atleast_1d(axes), worst.iterrows()):
            ax.imshow(overlay(face(w.path), cams["gradcam++"][i])); ax.axis("off")
            region = top.loc[(ds, w.files)].idxmax() if (ds, w.files) in top.index else "?"
            ax.set_title(f"{w.label}→{w.pred} {w.prob:.2f}\n{SHORT.get(region, region)}", fontsize=8)
        plt.suptitle(f"{name}: most confident mistakes on {ds} (truth→prediction, P(fake), top region)",
                     fontsize=10); plt.tight_layout(); plt.show()
    else:
        print(f"No mistakes among the analysed {ds} images.")

for backbone in ORDER:
    profile(backbone)

## 4 · Model selection for the explanation phases
Criteria are **fixed in advance and weighted equally**, and each is ranked across models (1 = best):
| Criterion | Measure | Better |
|---|---|---|
| Accuracy | FF++ test AUC, Celeb-DF test AUC (phase 1) | higher |
| Faithfulness | Grad-CAM++ gain over a random map: deletion (random − Grad-CAM++) and insertion (Grad-CAM++ − random), per dataset | higher |
| Face focus | share of Grad-CAM++ activation inside the face regions | higher |
| Cost | GFLOPs per image | lower |

The overall score is the **mean rank**. The faithfulness chart shows each model's gain with a 95% CI from paired
per-image differences; faded, hatched bars marked *n.s.* are **not** significantly better than random (Wilcoxon p ≥ 0.05).

In [ ]:
%matplotlib inline
from src.xai import model_selection
hits = find_all(os.path.join("compare", "summary.csv"))
GRADCAM_DIR, SUMMARY_CSV, SAVE_DIR = OUT, (hits[0] if hits else None), OUT
card, gains, focus, figs = model_selection.report(GRADCAM_DIR, SUMMARY_CSV, display=print, save_dir=SAVE_DIR)
plt.show()
display(Markdown("**Scorecard** (sorted by mean rank)"))
display(card.round(4))
display(Markdown(f"**Best overall by the pre-declared criteria: {card.index[0]}** (mean rank {card['mean rank'].iloc[0]:.2f}). "
                 f"Runner-up: {card.index[1]} ({card['mean rank'].iloc[1]:.2f})."))